In [ ]:
# GHCN SNWD Daily/Seasonal Validation (OL vs DA)

This notebook compares GEOSldas daily model snow depth (`SNODPLAND`) against the
station snow depth dataset built by:
`projects/GHCN_snwd/notebooks/ghcn_snwd_global_1998_present_build.ipynb`.

## Station Selection Rules (requested)

1. `n_valid_days >= 1500` over the analysis window.
2. `avg_snow_days_per_year >= 5`, where snow day means `snwd_mm > 0`.
3. Station must map in **both** OL and DA with:
   - `distance_km <= 40`
   - `abs(elev_diff_m) <= 500`


In [ ]:
# -------------------------
# Imports + configuration
# -------------------------
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display

SEASON_ORDER = ["DJF", "MAM", "JJA", "SON"]
EXCLUDE_JJA_FROM_ALL = True
EXCLUDE_DJF_FROM_ALL_FOR_SH = True  # If True and SH site, ALL excludes DJF instead of JJA.

EXPERIMENTS = {
    "OL": {
        "exp_name": "LS_OLv8_M36",
        "run_root": Path("/discover/nobackup/projects/land_da/M21C_land_sweeper/LS_OLv8_M36_v2/LS_OLv8_M36"),
    },
    "DA": {
        "exp_name": "LS_DAv8_M36",
        "run_root": Path("/discover/nobackup/projects/land_da/M21C_land_sweeper/LS_DAv8_M36_v3/LS_DAv8_M36"),
    },
}
DOMAIN = "SMAP_EASEv2_M36_GLOBAL"

ANALYSIS_START = "2000-01-01"
ANALYSIS_END = "2024-12-31"

# Requested thresholds
MIN_VALID_DAYS = 1500
MIN_AVG_SNOW_DAYS_PER_YEAR = 5.0
MAX_DISTANCE_KM = 40.0
MAX_ABS_ELEV_DIFF_M = 500.0

DISTANCE_METHOD = "haversine_km"  # or "squared_degree"
MAX_DISTANCE_DEG2 = 0.10

USE_RAW_TIMESERIES_CACHE = True
WRITE_RAW_TIMESERIES_CACHE = True
WRITE_COMBINED_LONG_PARQUET = False

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "common/python/io/read_GEOSldas.py").exists():
    for parent in REPO_ROOT.parents:
        if (parent / "common/python/io/read_GEOSldas.py").exists():
            REPO_ROOT = parent
            break

if not (REPO_ROOT / "common/python/io/read_GEOSldas.py").exists():
    raise FileNotFoundError("Could not find common/python/io/read_GEOSldas.py from current working directory")

PROJECT_ROOT = REPO_ROOT / "projects" / "GHCN_snwd"
NOTEBOOK_OUTPUT_DIR = PROJECT_ROOT / "outputs_ghcn_snwd_ol_da_validation"
NOTEBOOK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Output location used by ghcn_snwd_global_1998_present_build.ipynb
GHCN_BUILD_OUTDIR = REPO_ROOT / "ghcn_snwd_build" / "output"
GHCN_MANIFEST = GHCN_BUILD_OUTDIR / "ghcn_snwd_manifest.json"
GHCN_STATION_META = GHCN_BUILD_OUTDIR / "ghcn_snwd_station_metadata.parquet"
GHCN_STATION_COVERAGE = GHCN_BUILD_OUTDIR / "ghcn_snwd_station_coverage.parquet"
GHCN_PARQUET_DIR = GHCN_BUILD_OUTDIR / "ghcn_snwd_parquet"

_cache_tag = (
    f"{DOMAIN}_"
    f"{pd.Timestamp(ANALYSIS_START).strftime('%Y%m%d')}_"
    f"{pd.Timestamp(ANALYSIS_END).strftime('%Y%m%d')}"
)

RAW_TIMESERIES_NC = NOTEBOOK_OUTPUT_DIR / f"ghcn_snwd_raw_timeseries_{_cache_tag}.nc"
COMBINED_LONG_PARQUET = NOTEBOOK_OUTPUT_DIR / f"ghcn_snwd_obs_model_daily_joined_{_cache_tag}.parquet"

STATION_MAP_CSV = {
    k: NOTEBOOK_OUTPUT_DIR / f"ghcn_station_tile_map_{k}_{_cache_tag}.csv"
    for k in EXPERIMENTS.keys()
}

SELECTION_SUMMARY_PARQUET = NOTEBOOK_OUTPUT_DIR / f"ghcn_station_selection_summary_{_cache_tag}.parquet"
SELECTION_SUMMARY_CSV = NOTEBOOK_OUTPUT_DIR / f"ghcn_station_selection_summary_{_cache_tag}.csv"
ANNUAL_COUNTS_PARQUET = NOTEBOOK_OUTPUT_DIR / f"ghcn_station_year_counts_{_cache_tag}.parquet"

STATION_METRICS_CSV = NOTEBOOK_OUTPUT_DIR / f"ghcn_station_metrics_{_cache_tag}.csv"
STATION_METRICS_PARQUET = NOTEBOOK_OUTPUT_DIR / f"ghcn_station_metrics_{_cache_tag}.parquet"
DOMAIN_METRICS_CSV = NOTEBOOK_OUTPUT_DIR / f"ghcn_domain_metrics_{_cache_tag}.csv"
DOMAIN_METRICS_PARQUET = NOTEBOOK_OUTPUT_DIR / f"ghcn_domain_metrics_{_cache_tag}.parquet"
ELEVATION_MATCH_CSV = NOTEBOOK_OUTPUT_DIR / f"ghcn_station_tile_elevation_{_cache_tag}.csv"

sys.path.insert(0, str(REPO_ROOT / "common/python/io"))
from read_GEOSldas import read_tilecoord  # type: ignore

print(f"REPO_ROOT={REPO_ROOT}")
print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"GHCN_BUILD_OUTDIR={GHCN_BUILD_OUTDIR}")
print(f"GHCN_MANIFEST={GHCN_MANIFEST}")
print(f"GHCN_STATION_META={GHCN_STATION_META}")
print(f"GHCN_STATION_COVERAGE={GHCN_STATION_COVERAGE}")
print(f"GHCN_PARQUET_DIR={GHCN_PARQUET_DIR}")
print(f"DOMAIN={DOMAIN}")
print(f"ANALYSIS_START={ANALYSIS_START}")
print(f"ANALYSIS_END={ANALYSIS_END}")
print(f"MIN_VALID_DAYS={MIN_VALID_DAYS}")
print(f"MIN_AVG_SNOW_DAYS_PER_YEAR={MIN_AVG_SNOW_DAYS_PER_YEAR}")
print(f"MAX_DISTANCE_KM={MAX_DISTANCE_KM}")
print(f"MAX_ABS_ELEV_DIFF_M={MAX_ABS_ELEV_DIFF_M}")
print(f"EXCLUDE_JJA_FROM_ALL={EXCLUDE_JJA_FROM_ALL}")
print(f"EXCLUDE_DJF_FROM_ALL_FOR_SH={EXCLUDE_DJF_FROM_ALL_FOR_SH}")
print(f"RAW_TIMESERIES_NC={RAW_TIMESERIES_NC}")
for k, cfg in EXPERIMENTS.items():
    print(f"{k}: exp_name={cfg['exp_name']}, run_root={cfg['run_root']}")


In [ ]:
# -------------------------
# Helpers
# -------------------------
def haversine_km(lat1, lon1, lat2, lon2):
    # Vectorized Haversine distance in kilometers; lat2/lon2 can be arrays.
    r = 6371.0
    p1 = np.deg2rad(lat1)
    p2 = np.deg2rad(lat2)
    dphi = np.deg2rad(lat2 - lat1)
    dlambda = np.deg2rad(lon2 - lon1)
    a = np.sin(dphi / 2.0) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlambda / 2.0) ** 2
    return 2.0 * r * np.arcsin(np.sqrt(a))


def locate_tilecoord_file(run_root: Path, exp_name: str, domain: str, output_dir: Path) -> Path:
    candidates = [
        output_dir / "tilecoord.bin",
        output_dir / f"{exp_name}.ldas_tilecoord.bin",
        run_root / "output" / domain / "rc_out" / f"{exp_name}.ldas_tilecoord.bin",
        run_root / exp_name / "output" / domain / "rc_out" / f"{exp_name}.ldas_tilecoord.bin",
        run_root / f"{exp_name}.ldas_tilecoord.bin",
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError("Could not find tilecoord file. Checked: " + ", ".join([str(p) for p in candidates]))


def locate_daily_cat_file(run_root: Path, exp_name: str, domain: str, day: pd.Timestamp):
    y = f"Y{day.year:04d}"
    m = f"M{day.month:02d}"
    stamp = day.strftime("%Y%m%d")
    fname = f"{exp_name}.tavg24_1d_lnd_Nt.{stamp}_1200z.nc4"
    candidates = [
        run_root / "output" / domain / "cat" / "ens_avg" / y / m / fname,
        run_root / exp_name / "output" / domain / "cat" / "ens_avg" / y / m / fname,
        run_root / "cat" / "ens_avg" / y / m / fname,
    ]
    for p in candidates:
        if p.exists():
            return p
    return None


def _read_var_1d_tile(ds, name):
    x = ds[name]
    if "time" in x.dims:
        x = x.isel(time=0)
    return np.asarray(x.values, dtype=np.float32)


def read_daily_snwd_for_tiles(nc_path: Path, tile_indices: np.ndarray):
    # Read SNODPLAND (m), apply fill/physical checks, return mm.
    with xr.open_dataset(nc_path, decode_times=False) as ds:
        snod = _read_var_1d_tile(ds, "SNODPLAND")

    snod_sel_m = np.asarray(snod[tile_indices], dtype=np.float32)
    snod_sel_m[snod_sel_m > 1e14] = np.nan
    snod_sel_m[snod_sel_m < 0.0] = np.nan
    return snod_sel_m * 1000.0


def map_stations_to_tiles(
    tile_lat,
    tile_lon,
    tile_elev,
    station_df,
    distance_method="haversine_km",
    max_distance_deg2=0.1,
    max_distance_km=40.0,
):
    rows = []
    tile_lat = np.asarray(tile_lat, dtype=float)
    tile_lon = np.asarray(tile_lon, dtype=float)
    tile_elev = np.asarray(tile_elev, dtype=float)

    for _, r in station_df.iterrows():
        stn = str(r["station"])
        lat = float(r["station_lat"])
        lon = float(r["station_lon"])
        station_elev_m = float(r["station_elev_m"]) if np.isfinite(r["station_elev_m"]) else np.nan

        if not (np.isfinite(lat) and np.isfinite(lon)):
            continue

        if distance_method == "squared_degree":
            d_metric = (tile_lat - lat) ** 2 + (tile_lon - lon) ** 2
            j = int(np.argmin(d_metric))
            metric_val = float(d_metric[j])
            if max_distance_deg2 is not None and metric_val > float(max_distance_deg2):
                continue
            d_km = float(haversine_km(lat, lon, np.array([tile_lat[j]]), np.array([tile_lon[j]]))[0])
        elif distance_method == "haversine_km":
            d = haversine_km(lat, lon, tile_lat, tile_lon)
            j = int(np.argmin(d))
            d_km = float(d[j])
            if max_distance_km is not None and d_km > float(max_distance_km):
                continue
            metric_val = d_km
        else:
            raise ValueError(f"Unsupported distance_method={distance_method}")

        this_tile_elev = float(tile_elev[j]) if np.isfinite(tile_elev[j]) else np.nan
        elev_diff_m = np.nan
        if np.isfinite(this_tile_elev) and np.isfinite(station_elev_m):
            elev_diff_m = this_tile_elev - station_elev_m

        rows.append(
            {
                "station": stn,
                "station_lat": lat,
                "station_lon": lon,
                "station_elev_m": station_elev_m,
                "tile_index": j,
                "tile_lat": float(tile_lat[j]),
                "tile_lon": float(tile_lon[j]),
                "tile_elev_m": this_tile_elev,
                "elev_diff_m": elev_diff_m,
                "distance_km": d_km,
                "distance_metric": metric_val,
                "distance_method": distance_method,
            }
        )

    out_cols = [
        "station", "station_lat", "station_lon", "station_elev_m",
        "tile_index", "tile_lat", "tile_lon", "tile_elev_m", "elev_diff_m",
        "distance_km", "distance_metric", "distance_method",
    ]
    if len(rows) == 0:
        return pd.DataFrame(columns=out_cols)
    return pd.DataFrame(rows)[out_cols].sort_values("distance_km").reset_index(drop=True)


def build_obs_matrix(obs_df: pd.DataFrame, model_days: pd.DatetimeIndex, stations: list[str], value_col: str) -> np.ndarray:
    piv = obs_df.pivot_table(index="date", columns="station", values=value_col, aggfunc="mean")
    piv = piv.reindex(index=model_days, columns=stations)
    return np.asarray(piv.to_numpy(dtype=np.float32), dtype=np.float32)


def season_name(ts: pd.Timestamp) -> str:
    m = int(ts.month)
    if m in (12, 1, 2):
        return "DJF"
    if m in (3, 4, 5):
        return "MAM"
    if m in (6, 7, 8):
        return "JJA"
    return "SON"


def snow_error_metrics_ams(obs, mod, *, ddof_mean: int = 0):
    obs = np.asarray(obs, dtype=float)
    mod = np.asarray(mod, dtype=float)
    valid = np.isfinite(obs) & np.isfinite(mod)

    if not np.any(valid):
        return {"N": 0, "bias": np.nan, "rmse": np.nan, "ubrmse": np.nan, "nse": np.nan}

    o = obs[valid]
    m = mod[valid]
    e = m - o

    bias = float(np.mean(e))
    rmse = float(np.sqrt(np.mean(e ** 2)))

    em = e - np.mean(e)
    denom = max(len(em) - int(ddof_mean), 1)
    ubrmse = float(np.sqrt(np.sum(em ** 2) / denom))

    o_bar = float(np.mean(o))
    denom_nse = float(np.sum((o - o_bar) ** 2))
    if denom_nse <= 0:
        nse = np.nan
    else:
        nse = float(1.0 - (np.sum((m - o) ** 2) / denom_nse))

    return {"N": int(np.sum(valid)), "bias": bias, "rmse": rmse, "ubrmse": ubrmse, "nse": nse}


In [ ]:
# -------------------------
# Load GHCN build products
# -------------------------
if not GHCN_MANIFEST.exists():
    raise FileNotFoundError(f"Manifest not found: {GHCN_MANIFEST}")
if not GHCN_STATION_META.exists():
    raise FileNotFoundError(f"Station metadata not found: {GHCN_STATION_META}")
if not GHCN_STATION_COVERAGE.exists():
    raise FileNotFoundError(f"Station coverage not found: {GHCN_STATION_COVERAGE}")
if not GHCN_PARQUET_DIR.exists():
    raise FileNotFoundError(f"Observation parquet directory not found: {GHCN_PARQUET_DIR}")

with open(GHCN_MANIFEST, "r") as f:
    manifest = json.load(f)

station_meta = pd.read_parquet(GHCN_STATION_META).copy()
station_coverage = pd.read_parquet(GHCN_STATION_COVERAGE).copy()

station_meta["station_id"] = station_meta["station_id"].astype(str)
station_coverage["station_id"] = station_coverage["station_id"].astype(str)

station_meta["lat"] = pd.to_numeric(station_meta["lat"], errors="coerce")
station_meta["lon"] = pd.to_numeric(station_meta["lon"], errors="coerce")
station_meta["elev"] = pd.to_numeric(station_meta["elev"], errors="coerce")

print("Manifest keys:", sorted(manifest.keys()))
print(f"Station metadata rows: {len(station_meta):,}")
print(f"Station coverage rows: {len(station_coverage):,}")
print(f"Parquet root: {GHCN_PARQUET_DIR}")
display(station_meta.head(5))
display(station_coverage.head(5))


In [ ]:
# -------------------------
# Build station-year counts and apply observation filters
# -------------------------
analysis_start_ts = pd.Timestamp(ANALYSIS_START)
analysis_end_ts = pd.Timestamp(ANALYSIS_END)
analysis_years = list(range(analysis_start_ts.year, analysis_end_ts.year + 1))

annual_parts = []
for year in analysis_years:
    part_dir = GHCN_PARQUET_DIR / f"year={year}"
    if not part_dir.exists():
        print(f"Missing partition for year={year}; treating as zero observations for all stations.")
        continue

    df = pd.read_parquet(part_dir, columns=["station_id", "date", "snwd_mm"]).copy()
    if df.empty:
        continue

    df["station_id"] = df["station_id"].astype(str)
    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.normalize()
    df["snwd_mm"] = pd.to_numeric(df["snwd_mm"], errors="coerce")
    df = df[(df["date"] >= analysis_start_ts) & (df["date"] <= analysis_end_ts)].copy()
    if df.empty:
        continue

    grouped = (
        df.assign(snow_day=(df["snwd_mm"] > 0).astype(np.int16))
        .groupby("station_id", as_index=False)
        .agg(
            valid_days=("snwd_mm", "count"),
            snow_days=("snow_day", "sum"),
        )
        .rename(columns={"station_id": "station"})
    )
    grouped["year"] = int(year)
    annual_parts.append(grouped)

if len(annual_parts) == 0:
    raise RuntimeError("No observation rows found in configured analysis period")

annual_df = pd.concat(annual_parts, ignore_index=True)

all_stations = station_meta["station_id"].dropna().astype(str).unique().tolist()
full_index = pd.MultiIndex.from_product([all_stations, analysis_years], names=["station", "year"])
annual_full = (
    annual_df.set_index(["station", "year"])
    .reindex(full_index, fill_value=0)
    .reset_index()
)

annual_full["valid_days"] = pd.to_numeric(annual_full["valid_days"], errors="coerce").fillna(0).astype(int)
annual_full["snow_days"] = pd.to_numeric(annual_full["snow_days"], errors="coerce").fillna(0).astype(int)
annual_full["has_obs"] = annual_full["valid_days"] > 0

annual_full.to_parquet(ANNUAL_COUNTS_PARQUET, index=False)

station_summary = (
    annual_full.groupby("station", as_index=False)
    .agg(
        n_valid_days=("valid_days", "sum"),
        avg_snow_days_per_year=("snow_days", "mean"),
        n_years_with_obs=("has_obs", "sum"),
    )
)

station_summary = station_summary.merge(
    station_meta[["station_id", "lat", "lon", "elev", "name", "state", "gsn_flag", "hcn_crn_flag", "wmo_id"]],
    left_on="station",
    right_on="station_id",
    how="left",
)

station_summary = station_summary.rename(
    columns={
        "lat": "station_lat",
        "lon": "station_lon",
        "elev": "station_elev_m",
        "name": "station_name",
        "state": "station_state",
    }
)

coord_ok = (
    np.isfinite(pd.to_numeric(station_summary["station_lat"], errors="coerce"))
    & np.isfinite(pd.to_numeric(station_summary["station_lon"], errors="coerce"))
)
station_summary = station_summary.loc[coord_ok].copy()

obs_filtered_stations = station_summary.loc[
    (station_summary["n_valid_days"] >= int(MIN_VALID_DAYS))
    & (station_summary["avg_snow_days_per_year"] >= float(MIN_AVG_SNOW_DAYS_PER_YEAR))
].copy()

print(f"Stations with valid coordinates: {len(station_summary):,}")
print(f"Stations after obs filters: {len(obs_filtered_stations):,}")
print(f"Wrote annual station-year counts: {ANNUAL_COUNTS_PARQUET}")

display(obs_filtered_stations.sort_values(["n_valid_days", "avg_snow_days_per_year"], ascending=[False, False]).head(20))


In [ ]:
# -------------------------
# Map stations to GEOSldas tiles and apply mapping constraints
# -------------------------
if len(obs_filtered_stations) == 0:
    raise RuntimeError("No stations pass observation filters. Relax thresholds or inspect counts.")

site_maps = {}
for exp_key, cfg in EXPERIMENTS.items():
    exp_name = cfg["exp_name"]
    run_root = Path(cfg["run_root"])

    tilecoord_path = locate_tilecoord_file(run_root, exp_name, DOMAIN, NOTEBOOK_OUTPUT_DIR)
    tc = read_tilecoord(str(tilecoord_path))

    tile_lat_arr = np.asarray(tc["com_lat"], dtype=float)
    tile_lon_arr = np.asarray(tc["com_lon"], dtype=float)
    tile_elev_arr = np.asarray(tc["elev"], dtype=float) if "elev" in tc else np.full(tile_lat_arr.shape, np.nan, dtype=float)

    map_df = map_stations_to_tiles(
        tile_lat_arr,
        tile_lon_arr,
        tile_elev_arr,
        obs_filtered_stations[["station", "station_lat", "station_lon", "station_elev_m"]],
        distance_method=DISTANCE_METHOD,
        max_distance_deg2=MAX_DISTANCE_DEG2,
        max_distance_km=MAX_DISTANCE_KM,
    )

    map_df["abs_elev_diff_m"] = np.abs(pd.to_numeric(map_df["elev_diff_m"], errors="coerce"))
    map_df = map_df.loc[
        (pd.to_numeric(map_df["distance_km"], errors="coerce") <= float(MAX_DISTANCE_KM))
        & (map_df["abs_elev_diff_m"] <= float(MAX_ABS_ELEV_DIFF_M))
    ].copy()

    if len(map_df) == 0:
        raise RuntimeError(f"No stations remain after mapping constraints for {exp_key}")

    site_maps[exp_key] = map_df
    map_df.to_csv(STATION_MAP_CSV[exp_key], index=False)
    print(f"{exp_key}: mapped + constrained stations={len(map_df):,} -> {STATION_MAP_CSV[exp_key]}")

common_stations = sorted(set.intersection(*[set(df["station"]) for df in site_maps.values()]))
if len(common_stations) == 0:
    raise RuntimeError("No common stations across OL and DA after mapping constraints")

selected_station_df = (
    obs_filtered_stations[obs_filtered_stations["station"].isin(common_stations)]
    .sort_values(["n_valid_days", "avg_snow_days_per_year"], ascending=[False, False])
    .reset_index(drop=True)
)

selected_station_df.to_parquet(SELECTION_SUMMARY_PARQUET, index=False)
selected_station_df.to_csv(SELECTION_SUMMARY_CSV, index=False)

print(f"Final selected stations (common OL/DA): {len(selected_station_df):,}")
print(f"Wrote selection summary parquet: {SELECTION_SUMMARY_PARQUET}")
print(f"Wrote selection summary csv: {SELECTION_SUMMARY_CSV}")
display(selected_station_df.head(20))


In [ ]:
# -------------------------
# Build or load raw timeseries cache (obs + model SNODPLAND)
# -------------------------
loaded_from_cache = False
raw_ds = None
model_days = pd.date_range(pd.Timestamp(ANALYSIS_START), pd.Timestamp(ANALYSIS_END), freq="D")


def _cache_validation_errors(ds: xr.Dataset):
    errs = []
    required_vars = [
        "obs_snwd_mm",
        "model_snwd_mm",
        "station_lat",
        "station_lon",
        "tile_index",
        "tile_elev_m",
        "distance_km",
        "elev_diff_m",
    ]
    for v in required_vars:
        if v not in ds.data_vars:
            errs.append(f"missing variable: {v}")

    if str(ds.attrs.get("domain", "")) != str(DOMAIN):
        errs.append(f"domain mismatch: cache={ds.attrs.get('domain')} vs config={DOMAIN}")

    if str(ds.attrs.get("analysis_start", "")) != str(ANALYSIS_START):
        errs.append(f"analysis_start mismatch: cache={ds.attrs.get('analysis_start')} vs config={ANALYSIS_START}")

    if str(ds.attrs.get("analysis_end", "")) != str(ANALYSIS_END):
        errs.append(f"analysis_end mismatch: cache={ds.attrs.get('analysis_end')} vs config={ANALYSIS_END}")

    expected_exp = [str(k) for k in EXPERIMENTS.keys()]
    got_exp = [str(x) for x in ds.coords["exp"].values] if "exp" in ds.coords else []
    if got_exp != expected_exp:
        errs.append(f"exp coord mismatch: cache={got_exp} vs config={expected_exp}")

    if "time" not in ds.coords:
        errs.append("missing coord: time")
    else:
        got_time = pd.to_datetime(ds["time"].values)
        if len(got_time) != len(model_days):
            errs.append(f"time length mismatch: cache={len(got_time)} vs expected={len(model_days)}")

    if "station" not in ds.coords:
        errs.append("missing coord: station")
    elif int(ds.sizes.get("station", 0)) == 0:
        errs.append("station dimension is empty")

    return errs


if USE_RAW_TIMESERIES_CACHE and RAW_TIMESERIES_NC.exists():
    print(f"Found raw cache: {RAW_TIMESERIES_NC}")
    tmp_ds = xr.open_dataset(RAW_TIMESERIES_NC)
    cache_errs = _cache_validation_errors(tmp_ds)

    if len(cache_errs) == 0:
        raw_ds = tmp_ds
        loaded_from_cache = True
        print("Using existing raw cache (configuration match).")
    else:
        print("Existing raw cache does not match current configuration; re-extracting.")
        for msg in cache_errs:
            print(f"  - {msg}")
        try:
            tmp_ds.close()
        except Exception:
            pass
elif USE_RAW_TIMESERIES_CACHE:
    print(f"Raw cache not found: {RAW_TIMESERIES_NC}")
    print("Running extraction from daily cat files...")
else:
    print("USE_RAW_TIMESERIES_CACHE=False; running extraction from daily cat files...")


if not loaded_from_cache:
    station_list = selected_station_df["station"].astype(str).tolist()
    station_set = set(station_list)

    obs_parts = []
    for year in range(pd.Timestamp(ANALYSIS_START).year, pd.Timestamp(ANALYSIS_END).year + 1):
        part_dir = GHCN_PARQUET_DIR / f"year={year}"
        if not part_dir.exists():
            continue

        df = pd.read_parquet(part_dir, columns=["station_id", "date", "snwd_mm"]).copy()
        if df.empty:
            continue

        df["station_id"] = df["station_id"].astype(str)
        df = df[df["station_id"].isin(station_set)].copy()
        if df.empty:
            continue

        df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.normalize()
        df["snwd_mm"] = pd.to_numeric(df["snwd_mm"], errors="coerce")
        df = df[(df["date"] >= pd.Timestamp(ANALYSIS_START)) & (df["date"] <= pd.Timestamp(ANALYSIS_END))].copy()
        if df.empty:
            continue

        obs_parts.append(df.rename(columns={"station_id": "station"})[["station", "date", "snwd_mm"]])

    if len(obs_parts) == 0:
        raise RuntimeError("No observation rows found for selected stations")

    obs_daily = (
        pd.concat(obs_parts, ignore_index=True)
        .groupby(["station", "date"], as_index=False)
        .agg(snwd_mm=("snwd_mm", "mean"))
    )

    obs_snwd_mm = build_obs_matrix(obs_daily, model_days, station_list, "snwd_mm")

    exp_keys = list(EXPERIMENTS.keys())
    n_exp = len(exp_keys)
    n_time = len(model_days)
    n_station = len(station_list)

    model_snwd_mm = np.full((n_exp, n_time, n_station), np.nan, dtype=np.float32)
    tile_index = np.full((n_exp, n_station), -1, dtype=np.int32)
    tile_lat = np.full((n_exp, n_station), np.nan, dtype=np.float32)
    tile_lon = np.full((n_exp, n_station), np.nan, dtype=np.float32)
    tile_elev_m = np.full((n_exp, n_station), np.nan, dtype=np.float32)
    distance_km = np.full((n_exp, n_station), np.nan, dtype=np.float32)
    elev_diff_m = np.full((n_exp, n_station), np.nan, dtype=np.float32)

    for ei, exp_key in enumerate(exp_keys):
        cfg = EXPERIMENTS[exp_key]
        exp_name = cfg["exp_name"]
        run_root = Path(cfg["run_root"])

        m = site_maps[exp_key].set_index("station").loc[station_list].reset_index()
        tile_idx = m["tile_index"].to_numpy(dtype=int)

        tile_index[ei, :] = tile_idx.astype(np.int32)
        tile_lat[ei, :] = m["tile_lat"].to_numpy(dtype=np.float32)
        tile_lon[ei, :] = m["tile_lon"].to_numpy(dtype=np.float32)
        tile_elev_m[ei, :] = m["tile_elev_m"].to_numpy(dtype=np.float32)
        distance_km[ei, :] = m["distance_km"].to_numpy(dtype=np.float32)
        elev_diff_m[ei, :] = m["elev_diff_m"].to_numpy(dtype=np.float32)

        n_found = 0
        for ti, day in enumerate(model_days):
            f = locate_daily_cat_file(run_root, exp_name, DOMAIN, day)
            if f is None:
                continue

            snod_mm_vals = read_daily_snwd_for_tiles(f, tile_idx)
            model_snwd_mm[ei, ti, :] = snod_mm_vals
            n_found += 1

            if (ti + 1) % 365 == 0 or (ti + 1) == n_time:
                print(f"  {exp_key}: checked {ti + 1}/{n_time} days, files found={n_found}")

    station_meta_aligned = selected_station_df.set_index("station").loc[station_list].reset_index()

    raw_ds = xr.Dataset(
        data_vars={
            "obs_snwd_mm": (("time", "station"), obs_snwd_mm),
            "model_snwd_mm": (("exp", "time", "station"), model_snwd_mm),
            "station_lat": (("station",), station_meta_aligned["station_lat"].to_numpy(dtype=np.float32)),
            "station_lon": (("station",), station_meta_aligned["station_lon"].to_numpy(dtype=np.float32)),
            "station_elev_m": (("station",), station_meta_aligned["station_elev_m"].to_numpy(dtype=np.float32)),
            "tile_index": (("exp", "station"), tile_index),
            "tile_lat": (("exp", "station"), tile_lat),
            "tile_lon": (("exp", "station"), tile_lon),
            "tile_elev_m": (("exp", "station"), tile_elev_m),
            "distance_km": (("exp", "station"), distance_km),
            "elev_diff_m": (("exp", "station"), elev_diff_m),
        },
        coords={
            "exp": np.array(exp_keys, dtype=object),
            "time": pd.DatetimeIndex(model_days),
            "station": np.array(station_list, dtype=object),
        },
    )

    raw_ds.attrs["domain"] = DOMAIN
    raw_ds.attrs["analysis_start"] = str(ANALYSIS_START)
    raw_ds.attrs["analysis_end"] = str(ANALYSIS_END)
    raw_ds.attrs["distance_method"] = str(DISTANCE_METHOD)
    raw_ds.attrs["max_distance_km"] = float(MAX_DISTANCE_KM)
    raw_ds.attrs["max_abs_elev_diff_m"] = float(MAX_ABS_ELEV_DIFF_M)
    raw_ds.attrs["min_valid_days"] = int(MIN_VALID_DAYS)
    raw_ds.attrs["min_avg_snow_days_per_year"] = float(MIN_AVG_SNOW_DAYS_PER_YEAR)
    raw_ds.attrs["exp_keys"] = ",".join([str(k) for k in exp_keys])
    raw_ds.attrs["exp_names"] = ",".join([str(EXPERIMENTS[k]["exp_name"]) for k in exp_keys])

    if WRITE_RAW_TIMESERIES_CACHE:
        print(f"Writing raw cache: {RAW_TIMESERIES_NC}")
        raw_ds.to_netcdf(RAW_TIMESERIES_NC)
        print(f"Wrote raw cache: {RAW_TIMESERIES_NC}")

print(raw_ds)
print(f"Stations in raw dataset: {raw_ds.sizes['station']}")
print(f"Dates in raw dataset: {raw_ds.sizes['time']}")


In [ ]:
# -------------------------
# Optional combined long table (large)
# -------------------------
if WRITE_COMBINED_LONG_PARQUET:
    exp_vals = [str(x) for x in raw_ds["exp"].values]
    station_vals = [str(x) for x in raw_ds["station"].values]
    time_vals = pd.to_datetime(raw_ds["time"].values)

    obs_snwd_mm = np.asarray(raw_ds["obs_snwd_mm"].values, dtype=float)
    model_snwd_mm = np.asarray(raw_ds["model_snwd_mm"].values, dtype=float)

    recs = []
    for ei, exp_key in enumerate(exp_vals):
        tmp = pd.DataFrame(
            {
                "date": np.repeat(time_vals.values, len(station_vals)),
                "station": np.tile(np.array(station_vals, dtype=object), len(time_vals)),
                "experiment": exp_key,
                "obs_snwd_mm": obs_snwd_mm.reshape(-1),
                "model_snwd_mm": model_snwd_mm[ei, :, :].reshape(-1),
            }
        )
        recs.append(tmp)

    combined_long_df = pd.concat(recs, ignore_index=True)
    combined_long_df.to_parquet(COMBINED_LONG_PARQUET, index=False)
    print(f"Wrote combined long parquet: {COMBINED_LONG_PARQUET}")
else:
    print("WRITE_COMBINED_LONG_PARQUET=False; skipping combined long parquet write")


In [ ]:
# -------------------------
# Compute station and domain metrics
# -------------------------
exp_vals = [str(x) for x in raw_ds["exp"].values]
station_vals = [str(x) for x in raw_ds["station"].values]
time_vals = pd.to_datetime(raw_ds["time"].values)
season_vals = np.array([season_name(pd.Timestamp(t)) for t in time_vals], dtype=object)

obs_snwd_mm = np.asarray(raw_ds["obs_snwd_mm"].values, dtype=float)
model_snwd_mm = np.asarray(raw_ds["model_snwd_mm"].values, dtype=float)

station_lat = np.asarray(raw_ds["station_lat"].values, dtype=float)
station_lon = np.asarray(raw_ds["station_lon"].values, dtype=float)
station_elev_m = np.asarray(raw_ds["station_elev_m"].values, dtype=float)

tile_index = np.asarray(raw_ds["tile_index"].values)
tile_lat = np.asarray(raw_ds["tile_lat"].values, dtype=float)
tile_lon = np.asarray(raw_ds["tile_lon"].values, dtype=float)
tile_elev_m = np.asarray(raw_ds["tile_elev_m"].values, dtype=float)
distance_km = np.asarray(raw_ds["distance_km"].values, dtype=float)
elev_diff_m = np.asarray(raw_ds["elev_diff_m"].values, dtype=float)

# Build ALL mask by station so SH and NH can use different warm-season exclusions.
n_time = len(time_vals)
n_station = len(station_vals)
all_mask_by_station = np.ones((n_time, n_station), dtype=bool)

sh_station_mask = np.isfinite(station_lat) & (station_lat < 0.0)
nh_or_unknown_station_mask = ~sh_station_mask

if EXCLUDE_JJA_FROM_ALL and EXCLUDE_DJF_FROM_ALL_FOR_SH:
    if np.any(nh_or_unknown_station_mask):
        all_mask_by_station[:, nh_or_unknown_station_mask] &= (season_vals[:, None] != "JJA")
    if np.any(sh_station_mask):
        all_mask_by_station[:, sh_station_mask] &= (season_vals[:, None] != "DJF")
elif EXCLUDE_JJA_FROM_ALL:
    all_mask_by_station &= (season_vals[:, None] != "JJA")
elif EXCLUDE_DJF_FROM_ALL_FOR_SH and np.any(sh_station_mask):
    all_mask_by_station[:, sh_station_mask] &= (season_vals[:, None] != "DJF")

season_masks = {s: (season_vals == s) for s in SEASON_ORDER}

all_pairs = int(np.sum(all_mask_by_station))
max_pairs = int(n_time * n_station)
print(
    f"ALL mask included date-station slots: {all_pairs} / {max_pairs} "
    f"(EXCLUDE_JJA_FROM_ALL={EXCLUDE_JJA_FROM_ALL}, "
    f"EXCLUDE_DJF_FROM_ALL_FOR_SH={EXCLUDE_DJF_FROM_ALL_FOR_SH}, "
    f"SH stations={int(np.sum(sh_station_mask))})"
)

station_records = []
for ei, exp_key in enumerate(exp_vals):
    model_arr = model_snwd_mm[ei, :, :]

    for sj, stn in enumerate(station_vals):
        o_full = obs_snwd_mm[:, sj]
        m_full = model_arr[:, sj]

        for season_key in ["ALL", *SEASON_ORDER]:
            if season_key == "ALL":
                mask = all_mask_by_station[:, sj]
            else:
                mask = season_masks[season_key]

            o = o_full[mask]
            m = m_full[mask]
            metrics = snow_error_metrics_ams(o, m)

            valid = np.isfinite(o) & np.isfinite(m)
            obs_mean = float(np.mean(o[valid])) if np.any(valid) else np.nan
            model_mean = float(np.mean(m[valid])) if np.any(valid) else np.nan

            station_records.append(
                {
                    "station": stn,
                    "experiment": exp_key,
                    "variable": "SNODPLAND",
                    "units": "mm",
                    "season": season_key,
                    "obs_column": "obs_snwd_mm",
                    "model_column": "model_snwd_mm",
                    "N": metrics["N"],
                    "bias": metrics["bias"],
                    "rmse": metrics["rmse"],
                    "ubrmse": metrics["ubrmse"],
                    "nse": metrics["nse"],
                    "obs_mean": obs_mean,
                    "model_mean": model_mean,
                    "station_lat": station_lat[sj],
                    "station_lon": station_lon[sj],
                    "station_elev_m": station_elev_m[sj],
                    "tile_index": int(tile_index[ei, sj]) if np.isfinite(tile_index[ei, sj]) else -1,
                    "tile_lat": tile_lat[ei, sj],
                    "tile_lon": tile_lon[ei, sj],
                    "tile_elev_m": tile_elev_m[ei, sj],
                    "distance_km": distance_km[ei, sj],
                    "elev_diff_m": elev_diff_m[ei, sj],
                }
            )

station_metrics_df = pd.DataFrame(station_records)


domain_records = []
for ei, exp_key in enumerate(exp_vals):
    model_arr = model_snwd_mm[ei, :, :]

    for season_key in ["ALL", *SEASON_ORDER]:
        if season_key == "ALL":
            o = obs_snwd_mm[all_mask_by_station]
            m = model_arr[all_mask_by_station]
        else:
            mask = season_masks[season_key]
            o = obs_snwd_mm[mask, :].reshape(-1)
            m = model_arr[mask, :].reshape(-1)

        metrics = snow_error_metrics_ams(o, m)

        valid = np.isfinite(o) & np.isfinite(m)
        obs_mean = float(np.mean(o[valid])) if np.any(valid) else np.nan
        model_mean = float(np.mean(m[valid])) if np.any(valid) else np.nan

        domain_records.append(
            {
                "experiment": exp_key,
                "variable": "SNODPLAND",
                "units": "mm",
                "season": season_key,
                "obs_column": "obs_snwd_mm",
                "model_column": "model_snwd_mm",
                "N": metrics["N"],
                "bias": metrics["bias"],
                "rmse": metrics["rmse"],
                "ubrmse": metrics["ubrmse"],
                "nse": metrics["nse"],
                "obs_mean": obs_mean,
                "model_mean": model_mean,
            }
        )

domain_metrics_df = pd.DataFrame(domain_records)

station_metrics_df.to_csv(STATION_METRICS_CSV, index=False)
station_metrics_df.to_parquet(STATION_METRICS_PARQUET, index=False)
domain_metrics_df.to_csv(DOMAIN_METRICS_CSV, index=False)
domain_metrics_df.to_parquet(DOMAIN_METRICS_PARQUET, index=False)

elev_match_df = (
    station_metrics_df[station_metrics_df["season"] == "ALL"][
        [
            "station",
            "experiment",
            "station_lat",
            "station_lon",
            "station_elev_m",
            "tile_index",
            "tile_lat",
            "tile_lon",
            "tile_elev_m",
            "distance_km",
            "elev_diff_m",
        ]
    ]
    .drop_duplicates(["station", "experiment"])
    .sort_values(["experiment", "distance_km", "station"])
)
elev_match_df.to_csv(ELEVATION_MATCH_CSV, index=False)

print(f"Wrote station metrics CSV: {STATION_METRICS_CSV}")
print(f"Wrote station metrics parquet: {STATION_METRICS_PARQUET}")
print(f"Wrote domain metrics CSV: {DOMAIN_METRICS_CSV}")
print(f"Wrote domain metrics parquet: {DOMAIN_METRICS_PARQUET}")
print(f"Wrote elevation table CSV: {ELEVATION_MATCH_CSV}")

print("\nStation metrics shape:", station_metrics_df.shape)
print("Domain metrics shape:", domain_metrics_df.shape)

display(station_metrics_df.head(20))
display(domain_metrics_df.sort_values(["experiment", "season"]).head(20))
display(elev_match_df.head(20))


In [ ]:
# -------------------------
# Quick paired OL-vs-DA summary (ALL season)
# -------------------------
sub = station_metrics_df[
    (station_metrics_df["season"] == "ALL")
    & (station_metrics_df["variable"] == "SNODPLAND")
].copy()

piv = sub.pivot_table(index="station", columns="experiment", values=["nse", "rmse", "ubrmse", "bias"], aggfunc="mean")

rows = []
for metric in ["nse", "rmse", "ubrmse", "bias"]:
    if (metric, "OL") not in piv.columns or (metric, "DA") not in piv.columns:
        continue

    ol = pd.to_numeric(piv[(metric, "OL")], errors="coerce").to_numpy(dtype=float)
    da = pd.to_numeric(piv[(metric, "DA")], errors="coerce").to_numpy(dtype=float)
    valid = np.isfinite(ol) & np.isfinite(da)

    if not np.any(valid):
        rows.append({
            "metric": metric,
            "n_pair": 0,
            "mean_ol": np.nan,
            "mean_da": np.nan,
            "mean_improvement": np.nan,
        })
        continue

    ol_v = ol[valid]
    da_v = da[valid]

    if metric == "nse":
        improve = da_v - ol_v
    elif metric == "bias":
        improve = np.abs(ol_v) - np.abs(da_v)
    else:
        improve = ol_v - da_v

    rows.append(
        {
            "metric": metric,
            "n_pair": int(np.sum(valid)),
            "mean_ol": float(np.mean(ol_v)),
            "mean_da": float(np.mean(da_v)),
            "mean_improvement": float(np.mean(improve)),
        }
    )

paired_summary_df = pd.DataFrame(rows)
display(paired_summary_df)


## Plots: domain seasonal OL vs DA bars and delta improvement


In [ ]:
import matplotlib.pyplot as plt

EXP_OL    = "OL"
EXP_DA    = "DA"
OL_COLOR  = "#2878B5"
DA_COLOR  = "#F28E2B"
METRIC_ORDER = ["nse", "rmse", "ubrmse", "bias"]
METRIC_LABEL = {"nse": "NSE", "rmse": "RMSE", "ubrmse": "ubRMSE", "bias": "Bias"}
SEASONS_WITH_ALL = ["ALL"] + SEASON_ORDER
BOOT_N, BOOT_CI, BOOT_SEED = 1000, 95.0, 42

all_def_note = (
    "ALL excludes JJA for NH and DJF for SH"
    if (EXCLUDE_JJA_FROM_ALL and EXCLUDE_DJF_FROM_ALL_FOR_SH)
    else ("ALL excludes JJA" if EXCLUDE_JJA_FROM_ALL else "ALL includes all months")
)

FIG_DIR = NOTEBOOK_OUTPUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)


def _scale_to_mm(metric, x):
    arr = np.asarray(x, dtype=float)
    return arr * 1000.0 if metric != "nse" else arr


def _display_unit(metric):
    return "-" if metric == "nse" else "mm"


def _bootstrap_mean_ci(vals, n_boot=BOOT_N, ci=BOOT_CI, rng=None):
    x = np.asarray(vals, dtype=float)
    x = x[np.isfinite(x)]
    n = x.size
    if n == 0:
        return np.nan, np.nan, np.nan
    mu = float(np.mean(x))
    if n == 1:
        return mu, mu, mu
    if rng is None:
        rng = np.random.default_rng(BOOT_SEED)
    boot = np.array([float(np.mean(x[rng.integers(0, n, size=n)])) for _ in range(n_boot)])
    alpha = (100.0 - ci) / 2.0
    return mu, float(np.nanpercentile(boot, alpha)), float(np.nanpercentile(boot, 100.0 - alpha))


def _delta_series(paired_df, metric):
    ol = paired_df[EXP_OL].to_numpy(dtype=float)
    da = paired_df[EXP_DA].to_numpy(dtype=float)
    if metric == "nse":
        return da - ol
    if metric in {"rmse", "ubrmse"}:
        return ol - da
    if metric == "bias":
        return np.abs(ol) - np.abs(da)
    raise ValueError(metric)


rng = np.random.default_rng(BOOT_SEED)

# Bootstrap summaries

domain_boot_rows = []
for season_key in SEASONS_WITH_ALL:
    for exp_key in [EXP_OL, EXP_DA]:
        sub = station_metrics_df[
            (station_metrics_df["season"] == season_key)
            & (station_metrics_df["experiment"] == exp_key)
        ]
        for metric in METRIC_ORDER:
            vals = sub[metric].to_numpy(dtype=float)
            if metric == "bias":
                vals = np.abs(vals)
            mu, lo, hi = _bootstrap_mean_ci(vals, rng=rng)
            mu, lo, hi = _scale_to_mm(metric, mu), _scale_to_mm(metric, lo), _scale_to_mm(metric, hi)
            domain_boot_rows.append(
                {
                    "season": season_key,
                    "experiment": exp_key,
                    "metric": metric,
                    "mean": float(mu),
                    "ci_low": float(lo),
                    "ci_high": float(hi),
                }
            )

domain_boot_df = pd.DataFrame(domain_boot_rows)

delta_boot_rows = []
for season_key in SEASONS_WITH_ALL:
    for metric in METRIC_ORDER:
        sub = station_metrics_df[station_metrics_df["season"] == season_key][["station", "experiment", metric]].copy()
        p = sub.pivot_table(index="station", columns="experiment", values=metric, aggfunc="mean").reset_index()
        for col in [EXP_OL, EXP_DA]:
            if col not in p.columns:
                p[col] = np.nan
        delta_vals = _delta_series(p, metric)
        mu, lo, hi = _bootstrap_mean_ci(delta_vals, rng=rng)
        mu, lo, hi = _scale_to_mm(metric, mu), _scale_to_mm(metric, lo), _scale_to_mm(metric, hi)
        delta_boot_rows.append(
            {
                "season": season_key,
                "metric": metric,
                "delta_mean": float(mu),
                "ci_low": float(lo),
                "ci_high": float(hi),
            }
        )

delta_boot_df = pd.DataFrame(delta_boot_rows)


# Figure 1: OL vs DA bars
fig, axs = plt.subplots(1, len(METRIC_ORDER), figsize=(20, 5), sharex=True)
for mi, metric in enumerate(METRIC_ORDER):
    ax = axs[mi]
    tmp = domain_boot_df[domain_boot_df["metric"] == metric].copy()
    p = tmp.pivot_table(index="season", columns="experiment", values="mean", aggfunc="mean").reindex(SEASONS_WITH_ALL)
    ci_ol = tmp[tmp["experiment"] == EXP_OL].set_index("season").reindex(SEASONS_WITH_ALL)
    ci_da = tmp[tmp["experiment"] == EXP_DA].set_index("season").reindex(SEASONS_WITH_ALL)

    y_ol = p.get(EXP_OL, pd.Series(np.nan, index=SEASONS_WITH_ALL)).to_numpy(dtype=float)
    y_da = p.get(EXP_DA, pd.Series(np.nan, index=SEASONS_WITH_ALL)).to_numpy(dtype=float)

    yerr_ol = np.vstack([
        np.maximum(0.0, y_ol - ci_ol["ci_low"].to_numpy(dtype=float)),
        np.maximum(0.0, ci_ol["ci_high"].to_numpy(dtype=float) - y_ol),
    ])
    yerr_da = np.vstack([
        np.maximum(0.0, y_da - ci_da["ci_low"].to_numpy(dtype=float)),
        np.maximum(0.0, ci_da["ci_high"].to_numpy(dtype=float) - y_da),
    ])

    x = np.arange(len(SEASONS_WITH_ALL), dtype=float)
    w = 0.38
    ax.bar(
        x - w / 2,
        y_ol,
        width=w,
        color=OL_COLOR,
        label=EXP_OL,
        alpha=0.9,
        yerr=np.where(np.isfinite(yerr_ol), yerr_ol, np.nan),
        capsize=3,
        error_kw={"elinewidth": 0.9, "ecolor": "0.25"},
    )
    ax.bar(
        x + w / 2,
        y_da,
        width=w,
        color=DA_COLOR,
        label=EXP_DA,
        alpha=0.9,
        yerr=np.where(np.isfinite(yerr_da), yerr_da, np.nan),
        capsize=3,
        error_kw={"elinewidth": 0.9, "ecolor": "0.25"},
    )

    ax.set_xticks(x)
    ax.set_xticklabels(SEASONS_WITH_ALL)
    ax.grid(axis="y", alpha=0.25)
    ax.set_axisbelow(True)

    label = "|Bias|" if metric == "bias" else METRIC_LABEL[metric]
    ax.set_title(f"SNODPLAND | {label} ({_display_unit(metric)})")
    if mi == 0:
        ax.set_ylabel("Value")

axs[0].legend(loc="upper left")
fig.suptitle(
    "GHCN SNWD vs GEOSldas SNODPLAND — domain seasonal metrics (95% station-bootstrap CI)\n"
    + all_def_note
)
plt.tight_layout()
fig1_path = FIG_DIR / f"ghcn_domain_ol_da_bars_{_cache_tag}.png"
fig.savefig(fig1_path, dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Wrote: {fig1_path}")


# Figure 2: DA improvement bars (positive = DA improved)
fig, axs = plt.subplots(1, len(METRIC_ORDER), figsize=(20, 5), sharex=True)
for mi, metric in enumerate(METRIC_ORDER):
    ax = axs[mi]
    sub = delta_boot_df[delta_boot_df["metric"] == metric].set_index("season").reindex(SEASONS_WITH_ALL).reset_index()

    delta = sub["delta_mean"].to_numpy(dtype=float)
    lo = sub["ci_low"].to_numpy(dtype=float)
    hi = sub["ci_high"].to_numpy(dtype=float)
    yerr = np.vstack([
        np.maximum(0.0, delta - lo),
        np.maximum(0.0, hi - delta),
    ])

    x = np.arange(len(SEASONS_WITH_ALL), dtype=float)
    colors = np.where(delta >= 0, "#2ca02c", "#d62728")

    ax.bar(
        x,
        delta,
        color=colors,
        alpha=0.85,
        yerr=np.where(np.isfinite(yerr), yerr, np.nan),
        capsize=3,
        error_kw={"elinewidth": 0.9, "ecolor": "0.25"},
    )
    ax.axhline(0, color="k", linewidth=0.8, linestyle=":")
    ax.set_xticks(x)
    ax.set_xticklabels(SEASONS_WITH_ALL)
    ax.grid(axis="y", alpha=0.25)
    ax.set_axisbelow(True)

    label = "|Bias|" if metric == "bias" else METRIC_LABEL[metric]
    ax.set_title(f"Delta {label} (positive = DA improved) ({_display_unit(metric)})")
    if mi == 0:
        ax.set_ylabel("OL - DA (or DA - OL for NSE)")

fig.suptitle("GHCN SNWD — DA improvement over OL (95% station-bootstrap CI)\n" + all_def_note)
plt.tight_layout()
fig2_path = FIG_DIR / f"ghcn_delta_bars_{_cache_tag}.png"
fig.savefig(fig2_path, dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Wrote: {fig2_path}")


## Spatial map: per-station RMSE and NSE (ALL season)


In [ ]:
import warnings

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY = True
except ImportError:
    HAS_CARTOPY = False
    warnings.warn("cartopy not installed - skipping spatial map")

if HAS_CARTOPY:
    for exp_key in exp_vals:
        sub = station_metrics_df[
            (station_metrics_df["experiment"] == exp_key)
            & (station_metrics_df["season"] == "ALL")
        ].copy()
        sub["rmse_mm"] = sub["rmse"]

        fig, axs = plt.subplots(
            1,
            2,
            figsize=(20, 7),
            subplot_kw={"projection": ccrs.Robinson()},
        )

        for ax, (col, label, vmin, vmax, cmap) in zip(
            axs,
            [
                ("rmse_mm", "RMSE (mm)", 0, 200, "YlOrRd"),
                ("nse", "NSE", -1, 1, "RdYlGn"),
            ],
        ):
            ax.add_feature(cfeature.LAND, facecolor="0.92", zorder=0)
            ax.add_feature(cfeature.COASTLINE, linewidth=0.4)
            ax.add_feature(cfeature.BORDERS, linewidth=0.2)

            sc = ax.scatter(
                sub["station_lon"],
                sub["station_lat"],
                c=sub[col],
                cmap=cmap,
                vmin=vmin,
                vmax=vmax,
                s=6,
                transform=ccrs.PlateCarree(),
                zorder=3,
                linewidths=0,
            )

            plt.colorbar(
                sc,
                ax=ax,
                orientation="horizontal",
                pad=0.04,
                fraction=0.04,
                label=label,
            )
            ax.set_global()
            ax.set_title(f"{exp_key} - {label} (ALL season, GHCN stations)")

        plt.tight_layout()
        map_path = FIG_DIR / f"ghcn_station_map_{exp_key}_{_cache_tag}.png"
        fig.savefig(map_path, dpi=200, bbox_inches="tight")
        plt.show()
        plt.close(fig)
        print(f"Wrote: {map_path}")
